In [1]:
# Day 22 - Advanced RAG Techniques

!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

import torch
from transformers import pipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_1357/3489459964.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
# 1. Better Document Loading & Chunking
text = """
Ethiopia is one of the oldest countries in the world.
Addis Ababa is its capital and a major economic hub in Africa.
The country is known for its rich history, diverse cultures, and unique calendar.
Injera is the national dish. Coffee originated in Ethiopia.
Lalibela is famous for its rock-hewn churches, a UNESCO World Heritage Site.
"""

# Better chunking strategy
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

documents = text_splitter.create_documents([text])
print(f"Created {len(documents)} chunks with overlap")

Created 2 chunks with overlap


In [3]:
# 2. Embeddings + Vector Store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
# 3. Load LLM
pipe = pipeline("text-generation",
                model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device=0 if torch.cuda.is_available() else -1,
                max_new_tokens=300)

llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [5]:
# 4. Advanced RAG Prompt
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a knowledgeable Ethiopian AI assistant. Answer the question using only the provided context.

Context:
{context}

Question: {question}
Answer in a clear and friendly way:"""
)

# Simple RAG function
def advanced_rag_query(question):
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = prompt_template.format(context=context, question=question)
    response = llm.invoke(prompt)
    print("Q:", question)
    print("A:", response.strip())
    return response

# Test
advanced_rag_query("What is famous in Lalibela?")
advanced_rag_query("Tell me about Ethiopian food.")

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is famous in Lalibela?
A: You are a knowledgeable Ethiopian AI assistant. Answer the question using only the provided context.

Context:
Lalibela is famous for its rock-hewn churches, a UNESCO World Heritage Site.

Ethiopia is one of the oldest countries in the world. 
Addis Ababa is its capital and a major economic hub in Africa. 
The country is known for its rich history, diverse cultures, and unique calendar.
Injera is the national dish. Coffee originated in Ethiopia.

Question: What is famous in Lalibela?
Answer in a clear and friendly way:
Lalibela is famous for its rock-hewn churches, which are UNESCO World Heritage Sites.

Ethiopia is also famous for its rich history, diverse cultures, and unique calendar.

Injera is a staple Ethiopian dish made from teff flour, sorghum flour, and teff starch.

Coffee originated in Ethiopia, and it is a popular drink in the country.
Q: Tell me about Ethiopian food.
A: You are a knowledgeable Ethiopian AI assistant. Answer the question us

'You are a knowledgeable Ethiopian AI assistant. Answer the question using only the provided context.\n\nContext:\nEthiopia is one of the oldest countries in the world. \nAddis Ababa is its capital and a major economic hub in Africa. \nThe country is known for its rich history, diverse cultures, and unique calendar.\nInjera is the national dish. Coffee originated in Ethiopia.\n\nLalibela is famous for its rock-hewn churches, a UNESCO World Heritage Site.\n\nQuestion: Tell me about Ethiopian food.\nAnswer in a clear and friendly way:\nEthiopian cuisine is rich in flavors, spices, and ingredients. It features a variety of dishes, including soup, stew, salad, and dessert. The national dish is injera, a flatbread made out of teff flour, enriched with fermented teff flour. It is topped with a variety of vegetables and proteins, including meat and fish.\n\nLalibela is famous for its rock-hewn churches, each dating back to the 12th century. The churches are decorated with intricate carvings a